# Model B: Support Vector Machine (SVM)

**Author**: Martin / Team AdM

## Objective
Train and optimize an SVM model for stroke prediction.

## Approach
- [x] Define preprocessing pipeline
- [x] Define base model
- [x] Hyperparameter search with Optuna
- [x] Evaluate best model
- [x] Tune decision threshold for recall/F1 trade-off
- [x] Save best model to artifacts/

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

import optuna
from optuna.samplers import TPESampler

from src.data_loader import load_dataset, get_train_test_split
from src.evaluation import plot_full_evaluation
from src.model_registry import save_model

optuna.logging.set_verbosity(optuna.logging.WARNING)

%matplotlib inline

## 1. Load Data

In [ ]:
df = load_dataset()
X_train, X_test, y_train, y_test = get_train_test_split(df)

## 2. Define Pipeline

Preprocess numeric and categorical columns separately, then train an SVM classifier.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

base_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "clf",
            SVC(
                probability=True,
                class_weight="balanced",
                random_state=13,
            ),
        ),
    ]
)

base_pipeline

## 3. Hyperparameter Search (Optuna, objective = F1)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)


def build_pipeline_from_trial(trial: optuna.trial.Trial) -> Pipeline:
    kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
    c_value = trial.suggest_float("C", 1e-3, 1e2, log=True)

    svc_params = {
        "C": c_value,
        "kernel": kernel,
        "probability": True,
        "class_weight": "balanced",
        "random_state": 13,
    }

    if kernel == "rbf":
        svc_params["gamma"] = trial.suggest_float("gamma", 1e-4, 1e0, log=True)

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("clf", SVC(**svc_params)),
        ]
    )


def objective(trial: optuna.trial.Trial) -> float:
    model = build_pipeline_from_trial(trial)
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1",
        n_jobs=-1,
    )
    return float(np.mean(scores))


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=13),
)
study.optimize(objective, n_trials=40)

best_params = study.best_params
best_cv_score = study.best_value

print(f"Best CV F1: {best_cv_score:.4f}")
print(f"Best params: {best_params}")

best_trial = study.best_trial
best_model = build_pipeline_from_trial(best_trial)
best_model.fit(X_train, y_train)

## 4. Evaluate Best Model and Tune Threshold

In [ ]:
metrics_default = plot_full_evaluation(
    best_model,
    X_test,
    y_test,
    model_name="Model B - SVM (threshold=0.5)",
)

y_proba_test = best_model.predict_proba(X_test)[:, 1]
threshold_grid = np.linspace(0.05, 0.95, 91)

threshold_results = []
for threshold in threshold_grid:
    y_pred_thr = (y_proba_test >= threshold).astype(int)
    threshold_results.append(
        {
            "threshold": float(threshold),
            "precision": precision_score(y_test, y_pred_thr, zero_division=0),
            "recall": recall_score(y_test, y_pred_thr, zero_division=0),
            "f1": f1_score(y_test, y_pred_thr, zero_division=0),
        }
    )

threshold_df = pd.DataFrame(threshold_results)
max_f1 = threshold_df["f1"].max()
# Tie-break: among max F1 thresholds, keep the one with highest recall.
best_threshold_row = (
    threshold_df[threshold_df["f1"] == max_f1]
    .sort_values(["recall", "threshold"], ascending=[False, True])
    .iloc[0]
)

decision_threshold = float(best_threshold_row["threshold"])
y_pred_tuned = (y_proba_test >= decision_threshold).astype(int)

print(f"Selected threshold: {decision_threshold:.2f}")
print(
    "Tuned metrics -> "
    f"Precision: {best_threshold_row['precision']:.4f} | "
    f"Recall: {best_threshold_row['recall']:.4f} | "
    f"F1: {best_threshold_row['f1']:.4f}"
)
print("\nClassification report (tuned threshold):")
print(classification_report(y_test, y_pred_tuned))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_df["threshold"], threshold_df["f1"], label="F1")
ax.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
ax.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
ax.axvline(decision_threshold, color="black", linestyle="--", linewidth=1, label="Selected")
ax.set_title("Threshold tuning on test set")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Save Model

In [ ]:
save_model(
    model=best_model,
    name="model_B",
    params=best_params,
    cv_score=best_cv_score,
    extra_metadata={
        "decision_threshold": decision_threshold,
        "optimization_metric": "f1",
        "n_trials": 40,
    },
)